<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c36-header.png" alt="Nextia Learning · Text Classification and Routing: Send Each Message to the Right Place" width="100%">

# Solution: Fine-tune a small encoder on CPU

**[Fine-tune a small encoder on CPU](https://learning.nextia-ai.com/courses/text-routing/m04/fine-tune-a-small-encoder-on-cpu/)** · Text Classification and Routing: Send Each Message to the Right Place · Module 4, lesson 3 of 6 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** fine-tune a small multilingual encoder with a classification head on a laptop CPU, watch the validation loss, and load the ready checkpoint that the course trained for 3 epochs.

| | |
|---|---|
| **Time** | About 45 minutes. The base model download is about 470 MB, once (the model of the last two lessons), and the ready checkpoint 43 MB. |
| **Needs** | An internet connection for the setup cells and the downloads. No account and no key. The full training is optional: about 6 minutes on our laptop, longer on free Colab. |
| **You should know** | Sentence embeddings, from [Sentence embeddings for classification](https://learning.nextia-ai.com/courses/text-routing/m04/sentence-embeddings-for-classification/). |
| **Tested** | Python 3.12 with pandas 3.1.1, scikit-learn 1.9.1 and matplotlib 3.11.2, Transformers 5.22.2 and PyTorch 2.15.0 (CPU), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m04/fine-tune-a-small-encoder-on-cpu/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C36/M04-L03-fine-tune-a-small-encoder-on-cpu/fine-tune-a-small-encoder-on-cpu-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, scikit-learn 1.9.1, transformers 5.22.2, torch 2.15.0, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time. PyTorch is large (about 200 MB to download): the first run of this cell can take 2 minutes.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scikit-learn==1.9.1 matplotlib==3.11.2 transformers==5.22.2 torch==2.15.0
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)
import transformers; print("transformers", transformers.__version__)
import torch; print("torch", torch.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield inbox of this course (about 9 MB) into a folder `larkfield-inbox/data`, and the course's helper `routing.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 10000 messages (6000 train, 2000 validation, 2000 test), all synthetic.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-inbox` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield inbox is synthetic: we made it for this course. It has no real people. The answers of the hosted models were recorded once, so you need no key.

In [ ]:
# Setup: download the course data into larkfield-inbox/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/"
FILES = {
    "data/corrections.csv": "be3088aecd9b7ba5d07ea222b79e06a476806eba81bc80c2f33da695878e5cc0",
    "data/costs.json": "9a2d566f062c3382b8884ef07756f886179c69a3525d80e3d4c15c0c4fd4507e",
    "data/dataset-card.md": "ec1d7769d1d68a9e51a5b12660364c74352af936df79533a0d2170867171bdd1",
    "data/double_labelled.csv": "ba4a8dadc3c1ba8d48af893124bd33fb7a895f917a2add211486e3dbd353dfc9",
    "data/guidelines.md": "04df0cd72da4e6b7e6aa629ca21205e5395df7b66aae9268618ae14597e45782",
    "data/launch_labels_sample.csv": "600c689325b7c14e56f4a27e92e395eb0b2e965411b1abf9d724720c8b485029",
    "data/launch_month.csv": "7ef3cc2c5fa910023c6330c1e895744358846ce451e551b86be8c9abe1c4289f",
    "data/messages.csv": "266ca23c0551249fc114fc74253103ea8602a04be1fe31d1b7f91bc94ce6dd5e",
    "data/old_queue.csv": "e3cde5d19d3ebe2e6e0e067693c76796c0abbf89df0dc93dc310ad3fe42a5cae",
    "data/predictions/embeddings-newer.csv": "3c7ffe5b5371ca60c3b6bedc17b307142bcfd77f06c97db7ed267b703a50a8d2",
    "data/predictions/embeddings-newer.json": "dcd537f2971b9a48b5354cc9fbb48b87689b83b3a479da9acd119701c2fc19f6",
    "data/predictions/embeddings-older.csv": "a06c490ac17cbc1c3c75380f6cdf7775483de92961cdbb78b2a7914256926539",
    "data/predictions/embeddings-older.json": "db658bbd13861529122807726b6b41e1da9c70e3b55aba55cce9e868a795e8b4",
    "data/predictions/encoder-finetuned.csv": "abced2276671bfd413319573907a48c60952310e50978dba96c0dde0332b7d39",
    "data/predictions/encoder-finetuned.json": "1f7c89271a199c1d659bdc29d8ac0592558a1424218bd0519bbbde9840212b94",
    "data/predictions/gliclass-zero-shot.csv": "2c22e5036de94a376f7c0ccf22603c4d7c624135c8a4965123c9492748814008",
    "data/predictions/gliclass-zero-shot.json": "95ad2acafd48ad2c24516333dba7e78a3eabc7ca616c4053ce2f144f56cbaaec",
    "data/predictions/llm-few-shot.csv": "bcdaee3550f78ebb672465314bbcc9ad26283aaea69265a979739cf6a5f965e2",
    "data/predictions/llm-few-shot.json": "104b446894a0450a7e02fcd07c6db44f3e1a5486002b388374ce6dae4a744a12",
    "data/predictions/llm-zero-shot.csv": "5518e2b668a110b572687313bc13bd689774dff96c0a1614e1cd92e14ddf9ce7",
    "data/predictions/llm-zero-shot.json": "9a72ce600d0d858b00dd3b37247a305f9e8b3fc2fef3b8dbe60c0871b0c1b355",
    "data/predictions/local-llm-prompted.csv": "5bb0d84f8ee9b5086818d3525e1da3f8172b38b7875b10d1d3021b3ad5889f33",
    "data/predictions/local-llm-prompted.json": "ae328379d5ea294b3b59845ee143f24776787e748bc14c165b5e8e9ed4ff7158",
    "data/predictions/local-llm-scores.csv": "f85f570075ea9288fd14a6c3955d4310ba740814f3de472e8a263cd4f9682a5f",
    "data/predictions/local-llm-scores.json": "67a7e07ce065934a947f59c1c36d416eec76f8827f01500ce82ba6e6c4d9b99e",
    "data/predictions/nli-zero-shot.csv": "ecafb001370e5cf2904ce685683beafc86a2c290cd1da80438abb9912d099d3c",
    "data/predictions/nli-zero-shot.json": "54c2e724cb3c855a0fa6302d4ac0ab2df2ffaf788eec4ab198c6feb8d3508928",
    "data/prices.json": "9e98f9056d9d1e53d76a039da9b10d2b4fa54c8a5b35eb6cae32a2abdee7af38",
    "data/recordings/README.md": "d99eb50666045cc2ccf596e100103244fe1513b488fce46e26462a2666053ffc",
    "data/recordings/llm-few-shot.jsonl": "259a93f1d6dcbcb167cd45796c8e09a914e56c7741f14bd0aee53c996e8349d4",
    "data/recordings/llm-labels.jsonl": "cd6c670a23ee78cd3d8b51e5907ebae9d86448aa6236767c11d55227616fa8cc",
    "data/recordings/llm-reasoning.jsonl": "fc8ae631121f3240ce378df429af767ac52ea6e1560aa409de09ffc99b092300",
    "data/recordings/llm-zero-shot.jsonl": "957edfb8811e177c83bf23772f0da561659cf50728ffc95597f85cd7680aed86",
    "data/rules.json": "6f95d5e7b87869ec7975b198a3777cdd96a0db6a869554d922889479c87b2066",
    "routing.py": "8de6562d53415e5c3d8e74f44df9e8838cb71085011b29244187bde5a6f4d93c",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-inbox").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    local = HERE / name  # a copy next to the notebook, if there is one
    if local.exists():
        shutil.copy(local, path)
    else:
        download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-inbox and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
import csv
with open("data/messages.csv", encoding="utf-8") as f:
    splits = [row["split"] for row in csv.DictReader(f)]
print(f"Ready: {len(splits)} messages ({splits.count('train')} train, {splits.count('validation')} validation, "
      f"{splits.count('test')} test), all synthetic.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course helper as `rt`, reads the three parts of the inbox (`train`, `validation`, `test`), and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the course helper, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import routing as rt
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

messages = pd.read_csv("data/messages.csv")
train = messages[messages["split"] == "train"].reset_index(drop=True)
validation = messages[messages["split"] == "validation"].reset_index(drop=True)
test = messages[messages["split"] == "test"].reset_index(drop=True)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

## Setup: the ready checkpoint

Run the next cell. It downloads the course's fine-tuned encoder (43 MB) into `models/finetuned-encoder` and checks the checksums. It keeps only the weights that the training changed, in 16-bit numbers. You should see `Ready checkpoint: 2 files.`

In [ ]:
# Download the ready checkpoint and check it.
READY = {'models/finetuned-encoder/routes.json': '50afe5717b2f1ddd1ad6654fb68d7d96fc7766f6b3ea03984f977502adbcc3f7', 'models/finetuned-encoder/changed_weights.safetensors': '088e48f5198147eeb0b08be77f682c28822e91512420c694bad2c5693cfd5b63'}
for name, expected in READY.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    local = HERE / name  # a copy next to the notebook, if there is one
    if local.exists():
        shutil.copy(local, path)
    else:
        download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete it and run again.")
print(f"Ready checkpoint: {len(READY)} files.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The encoder and its new head

The base model is the older embedding model of this module. `AutoModelForSequenceClassification` adds a **classification head**: a small new layer that turns the model's 384 numbers into 7 scores, one per route. The head starts with random weights.

We **freeze** the word table: the part that gives each of the 250,000 tokens its first numbers. It is most of the model, and it does not need to change for our task. Everything else learns.

> **Predict.** About what share of the 118 million parameters will learn?

In [ ]:
import json, math, time, torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
torch.set_num_threads(4)
torch.manual_seed(36)
ROUTES = rt.ROUTES
meta = json.load(open("models/finetuned-encoder/routes.json"))
BASE, REV, MAX_LEN = meta["base_model"], meta["revision"], meta["max_length"]

def build():
    tok = AutoTokenizer.from_pretrained(BASE, revision=REV)
    model = AutoModelForSequenceClassification.from_pretrained(BASE, revision=REV, num_labels=len(ROUTES))
    model.base_model.embeddings.word_embeddings.weight.requires_grad = False  # freeze the word table
    return tok, model

tok, model = build()
total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{total:,} parameters, {trainable:,} learn ({trainable / total:.0%})")

> **Check.** You should see `117,656,455 parameters, 21,642,247 learn (18%)`.

## 2. A short training run: watch the loss

The **loss** measures how wrong the model's probabilities are: it is small when the model gives a high probability to the right route. We measure it on validation messages that the model does not learn from.

The next cell trains for 32 small steps on 1,024 training messages (32 messages per step). It takes about half a minute. It is a demonstration: the course's checkpoint trained for 564 steps.

In [ ]:
lab = {r: i for i, r in enumerate(ROUTES)}

def batches(texts, labels=None, size=32, shuffle=False, seed=36):
    order = np.arange(len(texts))
    if shuffle:
        np.random.default_rng(seed).shuffle(order)
    for i in range(0, len(order), size):
        j = order[i:i + size]
        enc = tok([texts[k] for k in j], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt")
        if labels is not None:
            enc["labels"] = torch.tensor([labels[k] for k in j])
        yield enc

@torch.no_grad()
def val_loss(model, texts, labels):
    model.eval()
    losses = [model(**enc).loss.item() * len(enc["labels"]) for enc in batches(texts, labels, size=64)]
    return sum(losses) / len(texts)

few = train.sample(1024, random_state=36)
x, y = list(few["text"]), [lab[r] for r in few["route"]]
vx, vy = list(validation["text"].head(300)), [lab[r] for r in validation["route"].head(300)]
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=6e-5, weight_decay=0.01)
print("validation loss before:", round(val_loss(model, vx, vy), 2))
t0 = time.time()
model.train()
for step, enc in enumerate(batches(x, y, shuffle=True), start=1):
    loss = model(**enc).loss
    loss.backward()
    opt.step()
    opt.zero_grad()
    if step % 8 == 0:
        print(f"step {step}: training loss {loss.item():.2f}")
print("validation loss after: ", round(val_loss(model, vx, vy), 2), f"({time.time() - t0:.0f} seconds)")

> **Check.** The validation loss starts near 1.95 (a random head: about the same probability for all 7 routes) and falls to about 1.3. Your numbers can differ a little.

The full run of the course went on for 3 epochs on all 6,000 messages, and its validation loss went down to 0.138. The lesson page shows the whole curve.

## 3. Optional: the full training

The next cell trains the whole model as the course did: 3 epochs, 564 steps. It took 327 seconds on our laptop (Apple M4, 4 threads); free Colab is slower. It runs only if you change `TRAIN_FULL` to `True`. Skip it and use the ready checkpoint in section 4: it is the result of this same code.

In [ ]:
TRAIN_FULL = False  # change to True to train for about 6 minutes or more
if TRAIN_FULL:
    tok, model = build()
    x, y = list(train["text"]), [lab[r] for r in train["route"]]
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=6e-5, weight_decay=0.01)
    steps = math.ceil(len(x) / 32) * 3
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 50) * max(0.0, 1 - s / steps))
    t0, step = time.time(), 0
    for epoch in range(3):
        model.train()
        for enc in batches(x, y, shuffle=True, seed=36 + epoch):
            model(**enc).loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            step += 1
        print(f"epoch {epoch + 1}: validation loss {val_loss(model, vx, vy):.3f} ({time.time() - t0:.0f} s)")
else:
    print("Skipped. Section 4 loads the ready checkpoint.")

## 4. Load the ready checkpoint

The ready checkpoint has only the weights that changed. We build the base model again, then put the changed weights on top. The word table comes from the base model, because it did not change.

In [ ]:
from safetensors.torch import load_file
tok, ready = build()
weights = {k: v.float() for k, v in load_file("models/finetuned-encoder/changed_weights.safetensors").items()}
missing, unexpected = ready.load_state_dict(weights, strict=False)
print("missing:", missing, "unexpected:", unexpected)
ready.eval()

@torch.no_grad()
def predict_proba(texts, size=64):
    out = []
    for enc in batches(list(texts), size=size):
        out.append(torch.softmax(ready(**enc).logits, -1).numpy())
    return np.concatenate(out)

t0 = time.time()
P = predict_proba(test["text"])
pred = np.array(ROUTES)[P.argmax(axis=1)]
print(f"{time.time() - t0:.0f} seconds for 2,000 messages")
rt.show([rt.score("fine-tuned encoder (ready checkpoint)", test, pred)])

> **Check.** `missing` lists only the word table (`...word_embeddings.weight`), and `unexpected` is empty. The table shows accuracy 0.98 (English 0.98, French 0.978) and 103.0 dollars per 1,000 messages. That is better than TF-IDF (0.962) and much better than the frozen embeddings (0.920), on this inbox.

## 5. Your turn: where the fine-tuning helps

The frozen embeddings + logistic regression of the first lesson are in the course data, as recorded predictions.

> **Your turn.** Count the test messages that the fine-tuned encoder routes correctly and the frozen embeddings route wrongly (`enc_only`), and the other way round (`emb_only`).

In [ ]:
frozen, _ = rt.predictions("embeddings-older", test)
truth = test["route"].to_numpy()
enc_right = pred == truth
emb_right = frozen["prediction"].to_numpy() == truth
enc_only = int((enc_right & ~emb_right).sum())
emb_only = int((emb_right & ~enc_right).sum())
print(enc_only, emb_only)

In [ ]:
expect('enc_only', enc_only, 132)

In [ ]:
expect('emb_only', emb_only, 14)

The fine-tuned encoder fixes many messages and breaks very few. Its layers learned what matters for Larkfield's routes, not only general meaning.

## 6. One message at a time

The router gets one message at a time. Measure the time per message that way.

In [ ]:
t0 = time.time()
for text in test["text"].head(100):
    predict_proba([text], size=1)
print(f"{(time.time() - t0) * 10:.1f} ms per message, one at a time")

> **Check.** On our laptop it was about 7 ms per message. Your number depends on your computer. It is slower than TF-IDF (about 0.1 ms) and still fast enough for an inbox of 65 messages a day.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m04/fine-tune-a-small-encoder-on-cpu/) has the loss curve of the full training, what each part of the model does, and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Newer embedding models](https://learning.nextia-ai.com/courses/text-routing/m04/newer-embedding-models/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/text-routing/m04/fine-tune-a-small-encoder-on-cpu/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.